# Lab 5: Reports and LLMs

**AI2Practice Research Lab** | about 1.5 hours

**The question.** How often do reports mention an incidental lung nodule, and can a large language model find them reliably?

By the end you will be able to treat radiology reports as data, write a simple keyword rule, compare it with saved answers from a large language model, and check that model's output case by case. You need Lab 3 (your `sens_spec()`). No API key or paid account is needed.

This lab has 3 parts: Part 1, Reports as text data; Part 2, Simple rules vs an LLM; Part 3, Checking LLM output.

**The scenario.** An **incidental pulmonary nodule** is a small round spot in the lung found on a scan done for another reason (a CT for chest pain, for example). Most are harmless, but some are early cancers, so guidelines such as the Fleischner Society's recommend follow-up depending on size and type. Nodules mentioned only in the body of a report are easily lost to follow-up. Hospitals therefore try to find them automatically in report text, increasingly with **large language models (LLMs)**: AI models such as Gemini, GPT or Claude that read and write text.

Your data: about 120 short, made-up chest CT reports. For each, a radiologist marked whether it describes a lung nodule (`truth_nodule`), and an LLM was asked the same question; its saved answers are in `llm_nodule`. Using saved answers keeps everyone's results identical and needs no account.

**Privacy.** Never paste real patient reports into a public AI tool. Real reports contain protected health information; hospitals use approved, private deployments for this work.

#### How the tasks work

Each task is tagged with a **mode** that tells you what kind of work it is:

| Mode | What you do |
|---|---|
| **Write** | Write a short piece of code in an empty cell. Try it on your own first; a hint and a hidden solution are there if you need them. |
| **Fill in** | Some code is written for you. Replace each `____` with the missing piece. |
| **Reuse** | Import a function you wrote in an earlier lab from your own `toolkit.py`. |
| **Direct the AI** | Ask an AI assistant to write the code, paste it in, run the test, and fill in a short AI-use log. |
| **Fix** | Code "written by an AI" has one bug. Find it and fix it. |
| **Read** | Read some code and predict what it does before you run it. |

Every task ends with a **test cell**. Run it after your code. If it prints `Check passed`, move on. If it shows an `AssertionError`, the message under it tells you what to look at; change your code and run both cells again.

## Set your GitHub username

Your practice data is generated from your GitHub username, so everyone gets a slightly different dataset and the lab page can check your own numbers. Type your username between the quotes (the part after `github.com/` in your profile address), then run the cell with **Shift + Enter**.

In [ ]:
GITHUB_USERNAME = "your-username"   # <- change this to your GitHub username

## Setup

The next cell gets everything ready. Run it once each time you open the notebook. It downloads two small helper files from the course repository if they are not here yet: `rl_data.py` makes your practice data, and `toolkit.py` is your own function library (more on that later). Lines starting with `!` are commands for the computer Colab runs on, not Python.

In [ ]:
# Setup: get the helper files (only if they are missing), then make your data
import os
COURSE = "https://raw.githubusercontent.com/UM-IHC-CA2i/CA2i-Research-Launchpad/main/lesson-notebooks"
FORK = f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/CA2i-Research-Launchpad/main/lesson-notebooks"
if not os.path.exists("rl_data.py"):
    !wget -q {COURSE}/rl_data.py
if not os.path.exists("toolkit.py"):
    # Your fork first, so functions you saved in earlier labs come back; else the course copy.
    !wget -q -O toolkit.py {FORK}/toolkit.py || wget -q -O toolkit.py {COURSE}/toolkit.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rl_data
import toolkit
from toolkit import save_to_toolkit

if GITHUB_USERNAME.strip().lower() in ("", "your-username"):
    print("Reminder: put your own GitHub username in the cell above, then run both cells again.")
rl_data.save(5, GITHUB_USERNAME, "reports.csv")   # your report collection
print("Ready. Data made for GitHub user:", rl_data.normalize_username(GITHUB_USERNAME))

---
## Part 1: Reports as text data

To a computer a report is a **string**: a sequence of characters. pandas has text tools under `.str` that work on a whole column at once: `.str.lower()` makes text lowercase, `.str.contains("word")` gives True where the word appears, `.str.split()` splits text into words. Load the reports and read a few.

In [ ]:
rep = pd.read_csv("reports.csv")
print("Reports:", len(rep))
for text in rep["text"].head(4):
    print("-", text, "\n")

**Mode: Read**

#### Task 1. A keyword and a negation

`in` checks whether one piece of text appears inside another. Predict what this gives, then set `prediction` to `True` or `False` (no quotes).

```python
"nodule" in "No pulmonary nodule."
```

In [ ]:
prediction = ____

<details><summary>Show the answer</summary>

`True`. The check only looks for the letters, so it is fooled by a negation: the sentence says there is **no** nodule. Keep this in mind for Part 2.

```python
prediction = True
```

</details>

In [ ]:
assert prediction is True, "The letters 'nodule' do appear in the sentence, so the answer is True, even though the sentence says there is no nodule."
print("Check passed")

**Mode: Fill in**

#### Task 2. How long are the reports?

Add a column `n_words` with the number of words in each report: split the text into words, then count them with `.str.len()`.

In [ ]:
rep["n_words"] = rep["text"].str.split().str.____()
print(rep["n_words"].describe().round(1))

<details><summary>Show a solution</summary>

```python
rep["n_words"] = rep["text"].str.split().str.len()
print(rep["n_words"].describe().round(1))
```

</details>

In [ ]:
assert rep["n_words"].iloc[0] == len(rep["text"].iloc[0].split()), "n_words should count the words in each report."
print("Check passed")

**Mode: Reuse**

#### Task 3. Describe the collection

Use `count_by()` (Lab 1) to count reports with and without a true nodule, and `describe_by()` (Lab 2) to compare their length. Store the number of reports with a true nodule in `n_truth`. If the import fails with "still a stub", your saved `toolkit.py` did not reach your fork. Open your Lab 1 notebook, copy your `count_by()` cell here, run it, and carry on (then save it with `save_to_toolkit(count_by)` and upload `toolkit.py` as in the Save to GitHub steps).

In [ ]:
import importlib
importlib.reload(toolkit)    # load the newest saved version of your toolkit
from toolkit import count_by, describe_by

truth_counts = ____
n_truth = truth_counts[1]
print(truth_counts, "\n")
describe_by(rep, "truth_nodule", ____)

<details><summary>Show a solution</summary>

```python
import importlib
importlib.reload(toolkit)    # load the newest saved version of your toolkit
from toolkit import count_by, describe_by

truth_counts = count_by(rep, "truth_nodule")
n_truth = truth_counts[1]
print(truth_counts, "\n")
describe_by(rep, "truth_nodule", "n_words")
```

</details>

In [ ]:
assert rl_data.check(5, GITHUB_USERNAME, "truth_n", n_truth), "n_truth should be the count of reports with truth_nodule == 1."
print("Reports with a true nodule:", n_truth, "of", len(rep))
print("Check passed")

---
## Part 2: Simple rules vs an LLM

Before LLMs, most report mining used **rules**: hand-written patterns such as "flag the report if it contains the word nodule". Rules are fast, free, transparent and private, but brittle: they miss other wordings and are fooled by negations. An LLM instead reads the sentence as a whole and can handle wording it has never seen, but it can also make confident mistakes, and its answers can change from run to run.

The saved LLM answers came from giving the model each report with this instruction:

In [ ]:
PROMPT = ("You are reading a chest CT report. Does it describe a lung nodule that is present now "
          "(including one called an opacity, lesion or density)? Answer only 1 for yes or 0 for no.\n\nReport: ")
print(PROMPT)

**Mode: Write**

#### Task 4. The keyword rule

Write the simplest possible rule: a column `rule_nodule` that is 1 when the report text contains `"nodule"` and 0 otherwise. Store how many reports it flags in `n_keyword`.

*Try this on your own first, without an AI assistant (you can switch off Colab's automatic code suggestions under Tools > Settings > AI assistance). Use the hint if you get stuck, and open the solution only after a real attempt.*

*Hint:* `rep["text"].str.contains("nodule")` gives True/False for each report; add `.astype(int)` to get 1/0. `.sum()` counts the 1s.

In [ ]:
# Your code here


print("Reports flagged by the keyword rule:", n_keyword)

<details><summary>Show a solution</summary>

```python
rep["rule_nodule"] = rep["text"].str.contains("nodule").astype(int)
n_keyword = rep["rule_nodule"].sum()

print("Reports flagged by the keyword rule:", n_keyword)
```

</details>

In [ ]:
assert set(rep["rule_nodule"]) <= {0, 1}, "rule_nodule should be 0 or 1 (use .astype(int))."
assert rl_data.check(5, GITHUB_USERNAME, "keyword_n", n_keyword), "n_keyword should be the number of reports containing 'nodule'."
print("Check passed")

**Mode: Reuse**

#### Task 5. Score the rule and the LLM with sens_spec()

Your `sens_spec()` from Lab 3 works for any yes/no prediction, not only AI scores. Use it to compare the keyword rule and the LLM with the truth. If the import fails with "still a stub", your saved `toolkit.py` did not reach your fork. Open your Lab 3 notebook, copy your `sens_spec()` cell here, run it, and carry on (then save it with `save_to_toolkit(sens_spec)` and upload `toolkit.py` as in the Save to GitHub steps).

In [ ]:
import importlib
importlib.reload(toolkit)    # load the newest saved version of your toolkit
from toolkit import sens_spec

rule = sens_spec(rep["truth_nodule"], rep["rule_nodule"])
llm = ____
compare = pd.DataFrame({"keyword rule": rule, "LLM": llm}).round(3)
compare

<details><summary>Show a solution</summary>

```python
import importlib
importlib.reload(toolkit)    # load the newest saved version of your toolkit
from toolkit import sens_spec

rule = sens_spec(rep["truth_nodule"], rep["rule_nodule"])
llm = sens_spec(rep["truth_nodule"], rep["llm_nodule"])
compare = pd.DataFrame({"keyword rule": rule, "LLM": llm}).round(3)
compare
```

</details>

In [ ]:
assert rl_data.check(5, GITHUB_USERNAME, "llm_fp", llm["fp"]), "llm should be sens_spec(truth_nodule, llm_nodule)."
assert rule["tp"] + rule["fp"] == rep["rule_nodule"].sum()
print("Check passed")

**Mode: Direct the AI**

#### Task 6. A better rule

Look at the reports the keyword rule gets wrong (`rep[rep["rule_nodule"] != rep["truth_nodule"]]["text"]`). Then ask an AI assistant to write a better rule as a function `better_rule(text)` that returns 1 or 0, handling negations ("No pulmonary nodule", "has resolved", "without nodule") and other words for a nodule ("opacity", "lesion", "density"). Apply it to make a column `better_nodule`. Paste, run, test, log.

Suggested prompt for your AI assistant (adapt it as you like):

> I have a pandas DataFrame `rep` with a column `text` holding short chest CT reports.
> Write a Python function better_rule(text) that returns 1 if the report describes a lung nodule that is present now,
> else 0. Count synonyms "opacity", "lesion", "density" as nodules. Ignore negated or resolved mentions such as
> "No pulmonary nodule.", "No suspicious pulmonary nodules", "without nodule", "has resolved". Work sentence by sentence.
> Then add a column rep["better_nodule"] with the result for every report.

In [ ]:
# Paste the AI's code here

<details><summary>Show a reference solution (only after trying with the AI)</summary>

```python
import re

def better_rule(text):
    """1 if any sentence describes a present nodule (or synonym), else 0."""
    for sentence in re.split(r"(?<=\.)\s+", text.lower()):
        if not re.search(r"\b(nodules?|opacity|lesion|density)\b", sentence):
            continue
        if sentence.startswith("no ") or "without" in sentence or "resolved" in sentence:
            continue
        return 1
    return 0

rep["better_nodule"] = rep["text"].apply(better_rule).astype(int)
```

</details>

In [ ]:
acc = lambda col: (rep[col] == rep["truth_nodule"]).mean()
assert set(rep["better_nodule"]) <= {0, 1}, "better_nodule should be 0 or 1."
assert acc("better_nodule") > acc("rule_nodule"), "The better rule should agree with the truth more often than the keyword rule."
print(f"Accuracy: keyword rule {acc('rule_nodule'):.1%}, better rule {acc('better_nodule'):.1%}, LLM {acc('llm_nodule'):.1%}")
print("Check passed")

**AI-use log (Task 6).** Double-click this cell to edit it.

- Tool I used:
- My prompt (paste it):
- What I changed in the AI's code, and why:
- Did the test cell pass on the first try? If not, what was wrong?

A careful rule can do very well on these reports, because they were built from a small set of sentences. Real reports are far more varied (typos, hedging such as "cannot exclude a small nodule", comparisons with older scans), which is where rules break and LLMs earn their keep. Either way, you only know how good a method is by checking it against a reference standard, as you just did.

---
## Part 3: Checking LLM output

LLMs make two kinds of mistakes worth naming. They can be simply wrong (a false positive or false negative), and they can **hallucinate**: state something fluent and confident that is not in the input at all. The only protection is to check their output against the source, systematically, before it is used for patient care or in a paper.

**Mode: Fix**

#### Task 7. Count the LLM's mistakes

An AI assistant wrote this code to count how often the LLM was wrong. The number it prints looks far too large. Find the one bug and fix it in place.

In [ ]:
# Written by an AI assistant: count the LLM's mistakes
mistakes = rep[rep["llm_nodule"] == rep["truth_nodule"]]
n_llm_errors = len(mistakes)
print("LLM errors:", n_llm_errors)

<details><summary>Show the bug and the fix</summary>

The filter uses `==`, which keeps the reports where the LLM **agrees** with the truth. Errors are where they differ, so it should be `!=`. A good sanity check: errors should equal FP + FN from `sens_spec()`.

```python
# Written by an AI assistant: count the LLM's mistakes (fixed)
mistakes = rep[rep["llm_nodule"] != rep["truth_nodule"]]
n_llm_errors = len(mistakes)
print("LLM errors:", n_llm_errors)
```

</details>

In [ ]:
assert n_llm_errors == llm["fp"] + llm["fn"], "An error is a report where the LLM answer differs from the truth: it should equal false positives + false negatives."
assert rl_data.check(5, GITHUB_USERNAME, "llm_errors", n_llm_errors)
print("Check passed")

**Mode: Direct the AI**

#### Task 8. Read every error

Counting errors is not enough; read them. Ask an AI assistant for code that builds a table `err_table` with one row per LLM error and the columns `report_id`, `error_type` (`"FP"` when the LLM said 1 but the truth is 0, `"FN"` for the reverse) and `text`, then prints each error. Read the errors: what kinds of sentences trip the LLM up?

Suggested prompt for your AI assistant (adapt it as you like):

> I have a pandas DataFrame `rep` with columns report_id, text, truth_nodule (0/1) and llm_nodule (0/1).
> Write code that makes a DataFrame err_table with one row per report where llm_nodule differs from truth_nodule,
> with columns report_id, error_type ("FP" if llm_nodule is 1 and truth is 0, "FN" otherwise) and text, in that order.
> Then print each row.

In [ ]:
# Paste the AI's code here

<details><summary>Show a reference solution (only after trying with the AI)</summary>

```python
err = rep[rep["llm_nodule"] != rep["truth_nodule"]].copy()
err["error_type"] = np.where(err["llm_nodule"] == 1, "FP", "FN")
err_table = err[["report_id", "error_type", "text"]]
for _, row in err_table.iterrows():
    print(row["report_id"], row["error_type"], "-", row["text"], "\n")
```

</details>

In [ ]:
assert list(err_table.columns)[:3] == ["report_id", "error_type", "text"], "err_table needs the columns report_id, error_type, text."
assert len(err_table) == n_llm_errors, "There should be one row per LLM error."
assert (err_table["error_type"] == "FP").sum() == llm["fp"], "Check how FP and FN are assigned."
print("Check passed")

**AI-use log (Task 8).** Double-click this cell to edit it.

- Tool I used:
- My prompt (paste it):
- What I changed in the AI's code, and why:
- Did the test cell pass on the first try? If not, what was wrong?

---
## Optional: try a live model

**Optional, not needed for the lab or the report back.** You can send one of these made-up reports to a live LLM and compare its answer with the saved one. Colab offers built-in access to Google's Gemini models from Python on many accounts (`from google.colab import ai`); if it is not available to you, the cell just says so. Other options are the Gemini side panel in Colab (paste the prompt and a report by hand) or another provider's Python package with your own API key, stored in Colab's **Secrets** panel (the key icon), never typed into the notebook. Only ever use made-up or properly de-identified text.

In [ ]:
# OPTIONAL: ask a live model about one report (may not be available on every account)
i = 0   # which report to try
try:
    from google.colab import ai
    answer = ai.generate_text(PROMPT + rep.loc[i, "text"])
    print("Live model said:", answer.strip())
    print("Saved LLM answer:", rep.loc[i, "llm_nodule"], " Truth:", rep.loc[i, "truth_nodule"])
except Exception as e:
    print("Colab's built-in model is not available here:", e)

### Putting it into words

Back to the question. How common were true nodules in your reports? How did the keyword rule and the LLM compare, and which mistakes did each make? Would you let the LLM decide on its own which patients get a follow-up letter, or use it to make a list that a person checks? Write one or two sentences on the lab page.

---
## Report back

You can now answer the question: how often do reports mention an incidental nodule, and can an LLM find them reliably? Run the cell below. It prints the numbers to enter on the Lab 5 page on the AI2Practice site, together with your GitHub username (use the same one there). If the page says a number is off, it gives you a hint first.

In [ ]:
REPORT = {
    "truth_n": int(n_truth),
    "keyword_n": int(n_keyword),
    "llm_errors": int(n_llm_errors),
}
print("Report back for Lab 5")
print("GitHub username:", rl_data.normalize_username(GITHUB_USERNAME))
LABELS = {
    "truth_n": "Reports with a true incidental nodule",
    "keyword_n": "Reports flagged by the rule \"nodule\" in text",
    "llm_errors": "LLM errors (false positives + false negatives)",
}
for key, value in REPORT.items():
    print(f"  {LABELS[key]}: {value}")

---
## Save to GitHub

**1. Save the notebook.** In Colab, choose **File > Save a copy in GitHub**. The first time, Colab asks you to sign in to GitHub and allow access. Then set:

- Repository: `<your-username>/CA2i-Research-Launchpad` (your fork)
- Branch: `main`
- File path: `lesson-notebooks/Lab_5_Reports_and_LLMs.ipynb`
- Commit message: `Lab 5 finished`

Leave "Include a link to Colab" ticked and click **OK**. A **commit** is a saved snapshot of your files with a short message, so you can always see what changed and go back.

That's it for Lab 5. Go back to the lab page, enter your report-back numbers and write one or two sentences on what you found.